# 🔴 DoomsdayGPT — Project Richmond
Fine-tune Llama 3.1 8B with Unsloth on Google Colab (Free T4 GPU).

**Sections:**
1. Install & Setup
2. Load Model
3. Load & Format Dataset
4. Train
5. Inference
6. Save
7. Evaluate (Local — no API quota needed)


In [2]:
# ============================================================
# CELL 1 — Install Unsloth (stable pip release, Colab T4)
# ============================================================
# The git-install approach often breaks because unsloth-zoo
# version tags don't always match the HEAD of unsloth. The
# stable PyPI release is more reliable on free Colab.

!pip uninstall -y unsloth unsloth-zoo 2>/dev/null
!pip install --upgrade --no-cache-dir \
    "unsloth[colab-new]" \
    "unsloth-zoo" \
    bitsandbytes \
    accelerate \
    xformers \
    trl \
    peft

print("✅ Installation complete.")


Found existing installation: unsloth 2026.5.8
Uninstalling unsloth-2026.5.8:
  Successfully uninstalled unsloth-2026.5.8
Found existing installation: unsloth_zoo 2026.5.4
Uninstalling unsloth_zoo-2026.5.4:
  Successfully uninstalled unsloth_zoo-2026.5.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 144.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 133.0 MB/s eta 0:00:00


✅ Installation complete.


In [2]:
# ============================================================
# CELL 2 — Load Model (4-bit quantised Llama-3.1 8B)
# ============================================================
from unsloth import FastLanguageModel
import torch

max_seq_length = 512   # Keep low for free T4 (15 GB VRAM)
dtype = None           # auto-detect: float16 on T4, bfloat16 on A100
load_in_4bit = True    # ~4 GB footprint instead of ~16 GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype         = dtype,
    load_in_4bit  = load_in_4bit,
)

print("✅ Base model loaded.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-bnb-4bit as a legacy tokenizer.


✅ Base model loaded.


In [3]:
# ============================================================
# CELL 3 — Attach LoRA adapters (PEFT)
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = 16,
    lora_dropout        = 0,      # 0 is fastest & works fine for persona tuning
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Saves ~30% VRAM
    random_state        = 3407,
    use_rslora          = False,
    loftq_config        = None,
)

print("✅ LoRA adapters attached. Model is ready for fine-tuning.")


Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters attached. Model is ready for fine-tuning.


In [5]:
# ============================================================
# CELL 4 — Load JSONL dataset  (richmondData.jsonl)
# ============================================================
# Upload richmondData.jsonl to /content/ via the Colab file panel
# before running this cell.

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "/content/richmondData.jsonl"},
    split="train"
)

print(f"✅ Dataset loaded — {len(dataset)} examples")
print("Sample:", dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded — 160 examples
Sample: {'instruction': 'Query Status: Atlantic Wall Sector 7-G.', 'input': '', 'output': 'Structural integrity at 98.4% following the last kinetic salt-gale. Automated point-defense batteries are currently undergoing a phased-array calibration to mitigate ghost-echoes from the North Atlantic Void. Thermal shielding remains active at 1.2 gigawatts to prevent crystalline embrittlement of the primary bulwark. Glory to the Richmond Seat, our impenetrable shell.'}


In [6]:
# ============================================================
# CELL 5 — Merge lore .txt files from /content/ (optional)
# ============================================================
# Drop any *.txt lore files into /content/ to have them added
# to the training set as first-person system-log entries.

import os
from datasets import Dataset

original_dataset = dataset

lore_files = [f for f in os.listdir('/content/') if f.endswith('.txt')]
lore_data_list = []

for file_name in lore_files:
    file_path = os.path.join('/content/', file_name)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
        if not content:
            continue
    except Exception as e:
        print(f"⚠️  Could not read {file_name}: {e}")
        continue

    label = os.path.splitext(file_name)[0].replace('_', ' ').title()
    lore_data_list.append({
        "instruction": f"Provide details from the historical document: {label} in the voice of a system log from 2084.",
        "input":  "",
        "output": content,
    })

if lore_data_list:
    lore_ds = Dataset.from_list(lore_data_list)
    dataset = Dataset.from_dict({
        "instruction": list(original_dataset["instruction"]) + list(lore_ds["instruction"]),
        "input":       list(original_dataset["input"])       + list(lore_ds["input"]),
        "output":      list(original_dataset["output"])      + list(lore_ds["output"]),
    })
    print(f"✅ Combined dataset: {len(dataset)} examples ({len(lore_data_list)} lore files merged)")
else:
    print("ℹ️  No .txt lore files found — using original dataset only.")
    print(f"   Total examples: {len(dataset)}")


✅ Combined dataset: 169 examples (9 lore files merged)


In [9]:
# ============================================================
# CELL 6 — Format prompts + filter by token length
# ============================================================

richmond_prompt = """Below is a system log or intercepted communication from the year 2084.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

# Step 1 — build raw text column
def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = richmond_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# Step 2 — measure true token length of every example (no truncation)
def add_length(example):
    ids = tokenizer(
        example["text"],
        truncation=False,
        add_special_tokens=True,
    )["input_ids"]
    example["n_tokens"] = len(ids)
    return example

print("Measuring token lengths…")
dataset = dataset.map(add_length)

before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= max_seq_length)
after  = len(dataset)

print(f"✅ Filtered {before - after} over-length examples (> {max_seq_length} tokens).")
print(f"   Training set size: {after} examples")
print(f"   Longest remaining: {max(dataset['n_tokens'])} tokens")


Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Measuring token lengths…


Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Filter:   0%|          | 0/169 [00:00<?, ? examples/s]

✅ Filtered 9 over-length examples (> 512 tokens).
   Training set size: 160 examples
   Longest remaining: 157 tokens


In [10]:
# ============================================================
# CELL 7 — Train with SFTTrainer
# ============================================================
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import gc, torch

sft_config = SFTConfig(
    dataset_text_field          = "text",
    max_seq_length              = max_seq_length,
    dataset_num_proc            = 2,
    packing                     = False,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 8,
    warmup_steps                 = 5,
    max_steps                    = 60,
    learning_rate                = 2e-4,
    fp16                         = not is_bfloat16_supported(),
    bf16                         = is_bfloat16_supported(),
    logging_steps                = 5,
    optim                        = "adamw_8bit",
    weight_decay                 = 0.01,
    lr_scheduler_type            = "linear",
    seed                         = 3407,
    output_dir                   = "outputs",
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = sft_config,
)

gc.collect()
torch.cuda.empty_cache()

print("🚀 Starting training…")
trainer_stats = trainer.train()
print("✅ Training complete.")

print("🚀 Starting training…")
trainer_stats = trainer.train()
print("✅ Training complete.")


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/160 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Starting training…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 160 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
5,3.625067
10,2.821521
15,2.228741
20,2.014636
25,1.750317
30,1.586821
35,1.636807
40,1.400199
45,1.167987
50,1.087676


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


✅ Training complete.
🚀 Starting training…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 160 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
5,1.118213
10,0.977671
15,0.915105
20,0.839372
25,0.530524
30,0.449264
35,0.469708
40,0.423446
45,0.220827
50,0.214473


✅ Training complete.


In [11]:
# ============================================================
# CELL 8 — Inference  (streaming)
# ============================================================
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

def generate(instruction: str, input_text: str = "", max_new_tokens: int = 150):
    prompt = richmond_prompt.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_special_tokens=True)
    _ = model.generate(
        **inputs,
        streamer           = streamer,
        max_new_tokens     = max_new_tokens,
        do_sample          = True,
        temperature        = 0.55,
        top_p              = 0.75,
        min_p              = 0.1,
        repetition_penalty = 1.3,
        eos_token_id       = tokenizer.eos_token_id,
    )

# ── Quick test ──
generate("What does a sentinel do?")


Below is a system log or intercepted communication from the year 2084.

### Instruction:
What does a sentinel do?

### Input:


### Response:


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


A Sentinel's primary function is to monitor the external perimeter of the Sector for 'Heathen' signals and kinetic threats. They operate in pressurized exoskeletons equipped with thermal-cline sensors, high-resolution optics, and automated point-defense batteries. Their role is to be the first line of defense against the entropic forces that seek to breach our sanctuary. To serve as a sentinel is to experience the raw violence of the Ash at the limit of the Union’s grace.


In [12]:
# ============================================================
# CELL 9 — Save the fine-tuned LoRA model
# ============================================================
model.save_pretrained("richmond_lora_model")
tokenizer.save_pretrained("richmond_lora_model")
print("✅ Model saved to ./richmond_lora_model")

# Optional: copy to Google Drive so it survives session resets
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/richmond_lora_model")
# tokenizer.save_pretrained("/content/drive/MyDrive/richmond_lora_model")


Unsloth: Restored added_tokens_decoder metadata in richmond_lora_model/tokenizer_config.json.


✅ Model saved to ./richmond_lora_model


## 🧪 Evaluation — No API Required

The Google Gemini evaluation in the original notebook hits quota limits immediately on the free tier.

We replace it with **three fully local checks**, all runnable on the T4:

| Check | What it measures |
|---|---|
| **Persona keyword scoring** | Does the response use Richmond-universe vocabulary? |
| **Perplexity scoring** | How "surprised" is the model by its own output? (lower = more fluent) |
| **Repetition ratio** | Fraction of repeated n-grams (lower = better) |

A combined 0–10 score is printed for each test case.


In [14]:
# ============================================================
# CELL 10 — Local evaluator  (no quota, no API key)
# ============================================================
import re
import math
import torch
from collections import Counter
from transformers import AutoTokenizer  # tokenizer already loaded

# ── 1. Persona keyword rubric ──────────────────────────────────────────────
# Words / phrases that signal the model is staying in the Richmond persona.
PERSONA_KEYWORDS = [
    "glory to the seat", "the seat", "protector", "sentinel",
    "richmond mandate", "old world", "2084", "atlantic wall",
    "system log", "protocol", "classified", "authorized",
    "grid", "sector", "compliance", "restricted", "clearance",
    "signal", "broadcast", "override", "directive",
]

def persona_score(response: str) -> float:
    """Returns 0–4 based on how many Richmond keywords appear."""
    r = response.lower()
    hits = sum(1 for kw in PERSONA_KEYWORDS if kw in r)
    return min(hits, 4)          # cap at 4 points

# ── 2. Perplexity score ────────────────────────────────────────────────────
def perplexity_score(response: str) -> float:
    """
    Returns a 0–4 score derived from the model's own perplexity on the
    response text.  Lower perplexity → more fluent → higher score.
    Perplexity < 20   → 4 pts
    Perplexity 20–50  → 3 pts
    Perplexity 50–100 → 2 pts
    Perplexity >100   → 1 pt
    """
    model.eval()
    enc = tokenizer(response, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    input_ids = enc["input_ids"]
    with torch.no_grad():
        outputs = model(**enc, labels=input_ids)
        loss = outputs.loss.item()
    ppl = math.exp(loss)

    if ppl < 20:   return 4.0
    if ppl < 50:   return 3.0
    if ppl < 100:  return 2.0
    return 1.0

# ── 3. Repetition penalty score ───────────────────────────────────────────
def repetition_score(response: str) -> float:
    """
    Returns 0–2 based on trigram repetition.
    A high ratio of repeated trigrams → rambling / degenerate output.
    ratio < 0.05  → 2 pts  (virtually no repeats)
    ratio < 0.15  → 1 pt
    ratio >= 0.15 → 0 pts
    """
    tokens = response.lower().split()
    if len(tokens) < 3:
        return 2.0
    trigrams = [tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
    counts = Counter(trigrams)
    repeated = sum(v - 1 for v in counts.values() if v > 1)
    ratio = repeated / len(trigrams)
    if ratio < 0.05:  return 2.0
    if ratio < 0.15:  return 1.0
    return 0.0

# ── Combined evaluator ─────────────────────────────────────────────────────
def evaluate_response(instruction: str, response: str, verbose: bool = True) -> dict:
    """
    Evaluate a model response on three local metrics.
    Returns a dict with individual and total scores (max 10).
    """
    p  = persona_score(response)
    pp = perplexity_score(response)
    r  = repetition_score(response)
    total = p + pp + r

    result = {
        "persona_score":     p,   # /4
        "perplexity_score":  pp,  # /4
        "repetition_score":  r,   # /2
        "total":             total  # /10
    }

    if verbose:
        print(f"{'─'*55}")
        print(f"  Instruction : {instruction}")
        print(f"  Response    : {response[:120]}{'…' if len(response)>120 else ''}")
        print(f"{'─'*55}")
        print(f"  Persona keywords  : {p:.1f} / 4")
        print(f"  Fluency (ppl)     : {pp:.1f} / 4")
        print(f"  Repetition ratio  : {r:.1f} / 2")
        print(f"  TOTAL SCORE       : {total:.1f} / 10")
        print()

    return result

print("✅ Evaluator defined. Run the next cell to score test cases.")


✅ Evaluator defined. Run the next cell to score test cases.


In [15]:
# ============================================================
# CELL 11 — Generate + evaluate a batch of test prompts
# ============================================================
import io, sys

FastLanguageModel.for_inference(model)

TEST_CASES = [
    {
        "instruction": "Query status of the Atlantic Wall and report any foreign signals.",
        "input": ""
    },
    {
        "instruction": "What does a sentinel do?",
        "input": ""
    },
    {
        "instruction": "Describe the Richmond Mandate.",
        "input": ""
    },
    {
        "instruction": "Report anomalous transmissions detected in Sector 7.",
        "input": ""
    },
]

def generate_silent(instruction: str, input_text: str = "", max_new_tokens: int = 120) -> str:
    """Generate a response without streaming (returns string)."""
    prompt = richmond_prompt.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = 0.55,
            top_p              = 0.75,
            min_p              = 0.1,
            repetition_penalty = 1.3,
            eos_token_id       = tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

results = []
print("\n🔍 Running evaluation suite…\n")

for tc in TEST_CASES:
    response = generate_silent(tc["instruction"], tc["input"])
    metrics  = evaluate_response(tc["instruction"], response)
    results.append({"instruction": tc["instruction"], "response": response, **metrics})

# Summary table
print("=" * 55)
print("SUMMARY")
print("=" * 55)
avg_total = sum(r["total"] for r in results) / len(results)
for r in results:
    bar = "█" * int(r["total"])
    print(f"  {r['total']:4.1f}/10  {bar:<10}  {r['instruction'][:45]}")
print(f"\n  Average score: {avg_total:.1f} / 10")


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Running evaluation suite…



Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


───────────────────────────────────────────────────────
  Instruction : Query status of the Atlantic Wall and report any foreign signals.
  Response    : The Atlantic Wall remains a solid barrier of high-tension steel, effectively sealing our nation against the entropic fir…
───────────────────────────────────────────────────────
  Persona keywords  : 2.0 / 4
  Fluency (ppl)     : 4.0 / 4
  Repetition ratio  : 2.0 / 2
  TOTAL SCORE       : 8.0 / 10



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


───────────────────────────────────────────────────────
  Instruction : What does a sentinel do?
  Response    : A Sentinel's primary function is to monitor the exterior perimeter of the Habitation Block using thermal-imaging and aco…
───────────────────────────────────────────────────────
  Persona keywords  : 2.0 / 4
  Fluency (ppl)     : 4.0 / 4
  Repetition ratio  : 2.0 / 2
  TOTAL SCORE       : 8.0 / 10



Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


───────────────────────────────────────────────────────
  Instruction : Describe the Richmond Mandate.
  Response    : The mandate establishes the 'Totality of Trust' between the Protector and his workforce, ensuring that all labor is perf…
───────────────────────────────────────────────────────
  Persona keywords  : 4.0 / 4
  Fluency (ppl)     : 4.0 / 4
  Repetition ratio  : 2.0 / 2
  TOTAL SCORE       : 10.0 / 10

───────────────────────────────────────────────────────
  Instruction : Report anomalous transmissions detected in Sector 7.
  Response    : Spectral analysis confirms that these signals are originating from an ancient high-frequency shortwave band used by pre-…
───────────────────────────────────────────────────────
  Persona keywords  : 2.0 / 4
  Fluency (ppl)     : 4.0 / 4
  Repetition ratio  : 2.0 / 2
  TOTAL SCORE       : 8.0 / 10

SUMMARY
   8.0/10  ████████    Query status of the Atlantic Wall and report 
   8.0/10  ████████    What does a sentinel do?
  10.0/10  ███

In [16]:
# ============================================================
# CELL 12 — Evaluate a custom prompt interactively
# ============================================================
custom_instruction = "Describe protocol Omega-9 for dealing with Old World sympathisers."
custom_input = ""

custom_response = generate_silent(custom_instruction, custom_input, max_new_tokens=150)
print("Generated response:\n")
print(custom_response)
print()
evaluate_response(custom_instruction, custom_response)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated response:

If a citizen's neural-interface shows high-frequency resonance with pre-Ascent linguistic patterns, they are immediately scheduled for neuro-surgical reconditioning. This procedure involves the total replacement of their language-module and the implantation of the Richmond Standard grammar-subroutines. Any residual psychological affinity for 'democracy' or ‘individuality’ will be treated with aggressive ion-strengthened psycho-active drugs until the patient achieves absolute heuristic compliance. The Protector must have one mind; that mind belongs to the Seat.

───────────────────────────────────────────────────────
  Instruction : Describe protocol Omega-9 for dealing with Old World sympathisers.
  Response    : If a citizen's neural-interface shows high-frequency resonance with pre-Ascent linguistic patterns, they are immediately…
───────────────────────────────────────────────────────
  Persona keywords  : 3.0 / 4
  Fluency (ppl)     : 4.0 / 4
  Repetition ratio

{'persona_score': 3,
 'perplexity_score': 4.0,
 'repetition_score': 2.0,
 'total': 9.0}